# Tags — Topic Tag Generation

Runs tag generation across the vault using the configured backend and compares
results against the ground truth in `data/ground_truth.yaml`.

In [13]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

from glob import glob
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.ground_truth import load_ground_truth
from notemine.tasks import run_tags

load_dotenv('../.env')
config = load_config('../config.toml')
gt = load_ground_truth('../' + config['vault']['ground_truth'].lstrip('./'))

vault = '../' + config['vault']['path'].lstrip('./')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

notes = sorted(
    p for p in glob(f'{vault}/**/*.md', recursive=True)
    if not Path(p).name.startswith('index_')
)
print(f'{len(notes)} notes found')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
20 notes found


## Run tag generation

In [14]:
import time

backend = config['run']['backend']

results = []
for path in notes:
    name = Path(path).name
    post = load_note(path)
    if len(post.content.strip()) < 50:
        continue

    gt_entry = gt.get(name, {})
    gt_tags = set(gt_entry.get('tags', []))

    row = {'file': name, 'lang': gt_entry.get('lang', '?'), 'ground_truth': sorted(gt_tags)}

    t0 = time.perf_counter()
    try:
        row[backend] = run_tags(backend, post.content, config)
    except Exception as e:
        row[backend] = f'ERROR: {e}'
    row['elapsed_s'] = time.perf_counter() - t0

    results.append(row)
    print(f'  {name}  ({row["elapsed_s"]:.1f}s)')

print(f'\nDone — {len(results)} notes processed')

  long_ai_gezondheidszorg_oncologie_onderzoek_nl.md  (4.9s)
  long_ai_healthcare_oncology_research_en.md  (3.2s)
  long_interview_climate_scientist_sea_level_en.md  (2.8s)
  long_interview_klimaatwetenschapper_zeespiegel_nl.md  (3.4s)
  long_opinie_eu_ai_act_digitaal_beleid_nl.md  (4.7s)
  long_opinion_eu_ai_act_digital_policy_en.md  (3.8s)
  medium_climate_blog_paris_agreement_en.md  (3.2s)
  medium_geschiedenis_gouden_eeuw_nl.md  (4.0s)
  medium_history_dutch_golden_age_en.md  (3.2s)
  medium_klimaat_blog_parijs_akkoord_nl.md  (3.9s)
  medium_reisgids_amsterdam_nl.md  (2.9s)
  medium_travel_guide_amsterdam_en.md  (2.5s)
  short_recensie_samsung_galaxy_s25_ultra_nl.md  (2.3s)
  short_recept_stamppot_nl.md  (2.5s)
  short_recipe_dutch_stamppot_en.md  (2.1s)
  short_review_samsung_galaxy_s25_ultra_en.md  (2.4s)
  short_sport_nieuws_ajax_champions_league_nl.md  (1.9s)
  short_sports_news_ajax_champions_league_en.md  (1.8s)
  short_tech_news_openai_announcement_en.md  (2.0s)
  short_tech_

In [15]:
import json

out_dir = Path('../results/tags')
out_dir.mkdir(parents=True, exist_ok=True)

for r in results:
    stem = Path(r['file']).stem
    payload = r[backend]
    data = {
        'file': r['file'],
        'backend': backend,
        'model': config[backend]['model'],
        'elapsed_s': round(r['elapsed_s'], 3),
        'ground_truth': r['ground_truth'],
        'tags': payload if not isinstance(payload, str) else [],
        'error': payload if isinstance(payload, str) else None,
    }
    (out_dir / f'{stem}.json').write_text(json.dumps(data, ensure_ascii=False, indent=2))

print(f'Saved {len(results)} files to {out_dir}')

Saved 20 files to ../results/tags


## Summary comparison

In [16]:
def tag_stats(gt_tags, pred):
    if isinstance(pred, str):
        return pred
    gt_set = set(gt_tags)
    pred_set = set(pred)
    matched = len(gt_set & pred_set)
    extra = len(pred_set - gt_set)
    pct = int(100 * matched / len(gt_set)) if gt_set else 0
    return f'{matched}/{len(gt_set)} ({pct}%)  +{extra} extra'

rows = []
for r in results:
    rows.append({
        'file': r['file'],
        'lang': r['lang'],
        'ground_truth': r['ground_truth'],
        backend: tag_stats(r['ground_truth'], r[backend]),
    })

pd.set_option('display.max_colwidth', 40)
pd.DataFrame(rows)

,file,lang,ground_truth,ollama
0,long_ai_gezondheidszorg_oncologie_on...,nl,"[ai, amsterdam-umc, cancer-diagnosis...",2/9 (22%) +6 extra
1,long_ai_healthcare_oncology_research...,en,"[ai, amsterdam-umc, cancer-diagnosis...",1/9 (11%) +4 extra
2,long_interview_climate_scientist_sea...,en,"[climate-science, delta-works, delta...",1/9 (11%) +4 extra
3,long_interview_klimaatwetenschapper_...,nl,"[climate-science, delta-works, delta...",1/9 (11%) +4 extra
4,long_opinie_eu_ai_act_digitaal_belei...,nl,"[big-tech, chatgpt, digital-policy, ...",2/9 (22%) +7 extra
5,long_opinion_eu_ai_act_digital_polic...,en,"[big-tech, chatgpt, digital-policy, ...",2/9 (22%) +6 extra
6,medium_climate_blog_paris_agreement_...,en,"[climate-change, cop30, emissions, f...",3/8 (37%) +5 extra
7,medium_geschiedenis_gouden_eeuw_nl.md,nl,"[17th-century, amsterdam, colonialis...",3/8 (37%) +7 extra
8,medium_history_dutch_golden_age_en.md,en,"[17th-century, amsterdam, colonialis...",5/8 (62%) +5 extra
9,medium_klimaat_blog_parijs_akkoord_n...,nl,"[climate-change, cop30, emissions, f...",3/8 (37%) +6 extra


## Fuzzy comparison

Same as above but a predicted tag counts as a match if `SequenceMatcher.ratio() >= 0.75` against any GT tag.
Reveals how much of the score gap is due to synonym/abbreviation mismatch vs genuinely wrong tags.

In [17]:
from difflib import SequenceMatcher

FUZZY_THRESHOLD = 0.75

def _fuzzy_match(tag: str, gt_set: set[str]) -> bool:
    return any(SequenceMatcher(None, tag, gt).ratio() >= FUZZY_THRESHOLD for gt in gt_set)

def fuzzy_tag_stats(gt_tags, pred):
    if isinstance(pred, str):
        return pred
    gt_set = set(gt_tags)
    pred_list = pred if not isinstance(pred, str) else []
    exact = len(gt_set & set(pred_list))
    fuzzy = sum(1 for t in pred_list if _fuzzy_match(t, gt_set))
    extra = len(set(pred_list) - gt_set)
    n = len(gt_set)
    return (
        f'exact {exact}/{n} ({int(100*exact/n) if n else 0}%)'
        f'  fuzzy {fuzzy}/{n} ({int(100*fuzzy/n) if n else 0}%)'
        f'  +{extra} extra'
    )

rows = []
for r in results:
    rows.append({
        'file': r['file'],
        'lang': r['lang'],
        backend: fuzzy_tag_stats(r['ground_truth'], r[backend]),
    })

pd.set_option('display.max_colwidth', 60)
pd.DataFrame(rows)

,file,lang,ollama
0,long_ai_gezondheidszorg_oncologie_onderzoek_nl.md,nl,exact 2/9 (22%) fuzzy 4/9 (44%) +6 extra
1,long_ai_healthcare_oncology_research_en.md,en,exact 1/9 (11%) fuzzy 1/9 (11%) +4 extra
2,long_interview_climate_scientist_sea_level_en.md,en,exact 1/9 (11%) fuzzy 2/9 (22%) +4 extra
3,long_interview_klimaatwetenschapper_zeespiegel_nl.md,nl,exact 1/9 (11%) fuzzy 3/9 (33%) +4 extra
4,long_opinie_eu_ai_act_digitaal_beleid_nl.md,nl,exact 2/9 (22%) fuzzy 2/9 (22%) +7 extra
5,long_opinion_eu_ai_act_digital_policy_en.md,en,exact 2/9 (22%) fuzzy 2/9 (22%) +6 extra
6,medium_climate_blog_paris_agreement_en.md,en,exact 3/8 (37%) fuzzy 4/8 (50%) +5 extra
7,medium_geschiedenis_gouden_eeuw_nl.md,nl,exact 3/8 (37%) fuzzy 3/8 (37%) +7 extra
8,medium_history_dutch_golden_age_en.md,en,exact 5/8 (62%) fuzzy 6/8 (75%) +5 extra
9,medium_klimaat_blog_parijs_akkoord_nl.md,nl,exact 3/8 (37%) fuzzy 3/8 (37%) +6 extra


In [22]:
g = ['house', 'fast', 'big', 'smart', 'happy']
p = ['home', 'quick', 'large', 'intelligent', 'joyful']
fuzzy_tag_stats(g, p)

'exact 0/5 (0%)  fuzzy 0/5 (0%)  +5 extra'

## Inspect a single note

Change `INSPECT` to any filename to see the full tag lists side by side.

In [18]:
INSPECT = Path(notes[0]).name

row = next((r for r in results if r['file'] == INSPECT), None)
if row is None:
    print(f'{INSPECT} not found')
else:
    gt_set = set(row['ground_truth'])
    pred = set(row[backend]) if not isinstance(row[backend], str) else set()

    all_tags = sorted(gt_set | pred)
    tag_rows = [{
        'tag': t,
        'ground_truth': '✓' if t in gt_set else '',
        backend: '✓' if t in pred else '',
    } for t in all_tags]

    print(f'=== {INSPECT} ===')
    display(pd.DataFrame(tag_rows))

=== long_ai_gezondheidszorg_oncologie_onderzoek_nl.md ===


,tag,ground_truth,ollama
0,ai,✓,
1,ai-pathology,,✓
2,amsterdam-umc,✓,✓
3,cancer-diagnosis,✓,
4,clinical-study,,✓
5,clinical-trial,✓,
6,deep-learning,✓,
7,diagnostic-accuracy,,✓
8,healthcare,✓,
9,healthcare-innovation,,✓


## (Optional) Save results to frontmatter

Uncomment to write results under `notemine.tags.<backend>`.

In [19]:
# from notemine.frontmatter_io import save_note
#
# for path in notes:
#     name = Path(path).name
#     row = next((r for r in results if r['file'] == name), None)
#     if row is None:
#         continue
#     post = load_note(path)
#     notemine = post.metadata.get('notemine', {})
#     tags_data = notemine.get('tags', {})
#     if not isinstance(row.get(backend), str):
#         tags_data[backend] = row[backend]
#     notemine['tags'] = tags_data
#     post.metadata['notemine'] = notemine
#     save_note(path, post)
# print('Saved.')